### Import required packages

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

### Check hardware

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using MPS: {torch.mps.is_available()}')

else:
    device = torch.device('cpu')
    print('Using CPU')

### Load HKR dataset

In [ ]:
from pathlib import Path
from torch.utils.data import random_split

from data.hkr_loader import validate_hkr_words_dataset, parse_hkr_words_dataset

hkr_words_path = Path("data/hkr/20200923_Dataset_Words_Public")

stats = validate_hkr_words_dataset(hkr_words_path)
print(f"OK: {hkr_words_path}")
print(f"Annotations: {stats['ann_count']}, images: {stats['img_count']}")

### Parse HKR dataset

In [ ]:
data_samples, missing_images = parse_hkr_words_dataset(
    hkr_words_path,
    skip_unmoderated=True,
    strip_text=True,
 )

print(f"Всего образцов: {len(data_samples)}, пропущено из-за отсутствия изображений: {missing_images}")

if len(data_samples) != 0:
    print(f"Пример образца: {data_samples[0]}")
else:
    print("Нет доступных образцов для обработки.")

### Inspect dataset

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def visualize_sample(sample):
    img_path = sample["image_path"]
    bbox = sample["bbox"]
    transcription = sample["transcription"]

    try:
        img = Image.open(img_path).convert("RGB")
        plt.imshow(img)
        plt.title(transcription)
        # plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Ошибка при загрузке изображения: {e}")

if data_samples:
    sample_idx = min(10000, len(data_samples) - 1)
    visualize_sample(data_samples[sample_idx])
else:
    print("Нет доступных образцов для визуализации.")


text_lengths = [len(sample["transcription"]) for sample in data_samples]

plt.hist(text_lengths, bins=30)
plt.xlabel("Длина текста")
plt.ylabel("Частота")
plt.title("Распределение длин строк")
plt.show()


all_chars = set()
for sample in data_samples:
    all_chars.update(sample["transcription"])
print(sorted(all_chars))


img_widths = []
img_heights = []
for sample in data_samples[:500]:
    img_path = sample["image_path"]
    try:
        img = Image.open(img_path)
        img_widths.append(img.width)
        img_heights.append(img.height)
    except Exception as e:
        print(f"Ошибка при загрузке изображения: {e}")

plt.hist(img_widths, bins=30, alpha=0.5, label="Ширина")
plt.hist(img_heights, bins=30, alpha=0.5, label="Высота")
plt.xlabel("Размер")
plt.ylabel("Частота")
plt.title("Распределение размеров изображений")
plt.legend()
plt.show()

### Encode text

In [ ]:
class CharacterMapper:
    def __init__(self, samples):
        unique_chars = set()
        for sample in samples:
            unique_chars.update(sample["transcription"])
        self.chars = sorted(unique_chars)

        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(self.chars)}
        self.idx_to_char = {idx + 1: char for idx, char in enumerate(self.chars)}
        self.idx_to_char[0] = ""

        self.num_classes = len(self.chars) + 1

    def encode(self, text):
        indices = []
        for char in text:
            if char in self.char_to_idx:
                indices.append(self.char_to_idx[char])
        return indices

    def decode(self, indices):
        chars = []
        prev_idx = None
        for idx in indices:
            if idx != 0 and idx != prev_idx and idx in self.idx_to_char:
                chars.append(self.idx_to_char[idx])
            prev_idx = idx
        return ''.join(chars)

mapper = CharacterMapper(data_samples)
print(f"Уникальных символов: {mapper.num_classes}")
test_text = "Шёл человек"
encoded = mapper.encode(test_text)
decoded = mapper.decode(encoded)
print(f"\nТест: '{test_text}' → {encoded} → '{decoded}'")

### Prepare dataset to load into CRNN

In [ ]:
from torch.utils.data import Dataset, DataLoader

class HKRDataset(Dataset):
    def __init__(self, samples, char_mapper, img_height=64):
        self.samples = samples
        self.char_mapper = char_mapper
        self.img_height = img_height

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample["image_path"]).convert("L")

        w, h = img.size
        new_w = int(self.img_height * w / h)
        img = img.resize((new_w, self.img_height), Image.Resampling.LANCZOS)

        img_array = np.array(img, dtype=np.float32) / 255.0
        img_array = (img_array - 0.5) / 0.5

        img_tensor = torch.FloatTensor(img_array).unsqueeze(0)

        target_ids = self.char_mapper.encode(sample["transcription"])
        target_tensor = torch.LongTensor(target_ids)

        return {
            "image": img_tensor,
            "target": target_tensor,
            "transcription": sample["transcription"],
            "img_path": sample["image_path"]
        }
    
def unify_size(batch):
    images = [item["image"] for item in batch]
    targets = [item["target"] for item in batch]
    texts = [item["transcription"] for item in batch]

    max_width = max(img.shape[2] for img in images)
    batch_size = len(images)
    height = images[0].shape[1]

    padded_images = torch.zeros(batch_size, 1, height, max_width)
    input_lengths = []

    for i, img in enumerate(images):
        w = img.shape[2]
        padded_images[i, :, :, :w] = img
        input_lengths.append(w)
    
    target_lengths = torch.LongTensor([len(t) for t in targets])
    targets_concatenated = torch.cat([t for t in targets])

    return {
        "images": padded_images,
        "targets": targets_concatenated,
        "target_lengths": target_lengths,
        "input_lengths": torch.LongTensor(input_lengths),
        "texts": texts
    }
    

dataset = HKRDataset(data_samples, mapper)
sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Original text: {sample['transcription']}")


train_ratio = 0.9
train_size = max(1, int(len(dataset) * train_ratio))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(52)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False,
                          collate_fn=unify_size, num_workers=0)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=unify_size, num_workers=0)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

batch = next(iter(train_loader))
print(f"Batch images shape: {batch['targets'].shape}")


## CRNN

In [ ]:
class CRNN(nn.Module):
    """CNN-BiLSTM-CTC"""
    
    def __init__(self, img_height=64, num_chars=80, hidden_size=256, num_layers=2):
        super(CRNN, self).__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(512, 512, kernel_size=2),
            nn.BatchNorm2d(512),
            nn.ReLU(),     
        )

        self.map_to_seq = nn.LazyLinear(hidden_size)

        self.rnn = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=0.3 if num_layers > 1 else 0,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size * 2, num_chars + 1)

    def forward(self, x):
        conv = self.cnn(x)
        b, c, h, w = conv.size()

        conv = conv.permute(0, 3, 1, 2).reshape(b, w, c * h)

        seq = self.map_to_seq(conv)
        rnn_out, _ = self.rnn(seq)
        output = self.fc(rnn_out)
        return torch.nn.functional.log_softmax(output, dim=2)
    
    
model = CRNN(
    img_height=64,
    num_chars=mapper.num_classes,
    hidden_size=256,
    num_layers=2
)
model = model.to(device)

with torch.no_grad():
    _ = model(torch.randn(1, 1, 64, 256).to(device))

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params}")

### Decode and evaluate

In [ ]:
def decode_predictions(outputs, char_mapper):
    _, max_indices = torch.max(outputs, dim=2)
    predictions = []
    for seq in max_indices:
        text = char_mapper.decode(seq.cpu().numpy())
        predictions.append(text)
    return predictions


def ctc_input_lengths_from_padded_images(widths, max_t):
    lens = widths // 4 - 1
    lens = torch.clamp(lens, min=1, max=max_t)
    return lens


def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]


def calculate_CER(prediction, ground_truth):
    if len(ground_truth) == 0:
        return 0.0 if len(prediction) == 0 else 1.0
    return levenshtein_distance(prediction, ground_truth) / len(ground_truth)

def calculate_WER(prediction, ground_truth):
    pred_words = prediction.split()
    gt_words = ground_truth.split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return levenshtein_distance(pred_words, gt_words) / len(gt_words)


def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = len(train_loader)

    for batch_idx, batch in enumerate(train_loader):
        images = batch["images"].to(device)
        targets = batch["targets"].to(device)
        target_lengths = batch["target_lengths"].to(device)
        raw_widths = batch["input_lengths"].to(device)

        outputs = model(images)
        input_lengths = ctc_input_lengths_from_padded_images(raw_widths, outputs.size(1))

        outputs_ctc = outputs.permute(1, 0, 2)
        loss = criterion(outputs_ctc, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == num_batches:
            print(f"Batch {batch_idx + 1}/{num_batches}, Loss: {loss.item():.4f}")

    return total_loss / max(1, num_batches)


def validate(model, val_loader, criterion, char_mapper, device):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_ground_truths = []
    total_cer = 0.0
    total_wer = 0.0
    num_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["images"].to(device)
            targets = batch["targets"].to(device)
            target_lengths = batch["target_lengths"].to(device)
            raw_widths = batch["input_lengths"].to(device)
            texts = batch["texts"]

            outputs = model(images)
            input_lengths = ctc_input_lengths_from_padded_images(raw_widths, outputs.size(1))

            outputs_ctc = outputs.permute(1, 0, 2)
            loss = criterion(outputs_ctc, targets, input_lengths, target_lengths)
            total_loss += loss.item()
            predictions = decode_predictions(outputs, char_mapper)
            all_predictions.extend(predictions)
            all_ground_truths.extend(texts)

            for pred, gt in zip(predictions, texts):
                total_cer += calculate_CER(pred, gt)
                total_wer += calculate_WER(pred, gt)
                num_samples += 1

    avg_loss = total_loss / max(1, len(val_loader))
    avg_cer = total_cer / max(1, num_samples)
    avg_wer = total_wer / max(1, num_samples)
    return avg_loss, avg_cer, avg_wer, all_predictions, all_ground_truths



# print(f"Пример расстояния Левенштейна: '{test_text}' vs 'Helo my friend' → {levenshtein_distance(test_text, 'Helo my friend')}")

In [ ]:
import torch.optim as optim

if "dataset" not in globals():
    if "data_samples" not in globals() or "mapper" not in globals():
        raise ValueError("Нет данных для создания датасета. Запустите верхние ячейки кода")
    dataset = HKRDataset(data_samples, mapper)

if "train_dataset" not in globals() or "val_dataset" not in globals():
    train_ratio = 0.9
    train_size = max(1, int(len(dataset) * train_ratio))
    val_size = len(dataset) - train_size

    generator = torch.Generator().manual_seed(52)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

batch_size = 6
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=unify_size,
    num_workers=0,
    pin_memory=(device.type == 'cuda')
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=unify_size,
    num_workers=0,
    pin_memory=(device.type == 'cuda')
)

print(f"Batch size: {batch_size}")
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

In [ ]:
EPOCHS = 12
LEARNING_RATE = 1e-4
PHASE2_LEARNING_RATE = 5e-5
PHASE2_START_EPOCH = 11
PATIENCE = 12

model = model.to(device)

resume_candidates = [
    Path("crnn_checkpoint_best_ru_1004.pth"),
    Path("crnn_checkpoint_last_ru_1004.pth")
]

resume_path = next((p for p in resume_candidates if p.exists()), None)

if resume_path is not None:
    try:
        print(f"Загрузка контрольной точки из: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device)
        state_dict = checkpoint.get("model_state_dict") if isinstance(checkpoint, dict) else checkpoint
        if state_dict is not None:
            model.load_state_dict(state_dict)
            print("Контрольная точка успешно загружена.")
        else:
            exception = ValueError("Неверный формат контрольной точки: отсутствует 'model_state_dict'")
            print(f"Ошибка загрузки контрольной точки: {exception}")
    except Exception as e:
        print(f"Ошибка при загрузке контрольной точки: {e}")
else:
    print("Контрольная точка не найдена. Начало обучения с нуля.")


criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

history = {
    "train_loss": [],
    "val_loss": [],
    "val_cer": [],
    "val_wer": [],
    "best_val_cer": float('inf'),
    "best_epoch": 0
}

best_checkpoint_path = Path("crnn_checkpoint_best_ru_1004.pth")
last_checkpoint_path = Path("crnn_checkpoint_last_ru_1004.pth")
epochs_no_improve = 0

fixed_batch = next(iter(val_loader))
fixed_images = fixed_batch['images'].to(device)
fixed_texts = fixed_batch['texts']

print(f"Начало обучения... Устройство: {device}")
print(f"Этап 1: epoch 1-{PHASE2_START_EPOCH - 1}, lr={LEARNING_RATE}")
print(f"Этап 2: epoch {PHASE2_START_EPOCH}-{EPOCHS}, lr={PHASE2_LEARNING_RATE}")

for epoch in range(1, EPOCHS + 1):
    if epoch == PHASE2_START_EPOCH:
        for pg in optimizer.param_groups:
            pg['lr'] = PHASE2_LEARNING_RATE
        print(f"\nПереключение на этап 2. Новый lr = {PHASE2_LEARNING_RATE}")

    print("\n" + "="*30)
    print(f"Эпоха {epoch}/{EPOCHS}")
    print("="*30)

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    history["train_loss"].append(train_loss)
    print(f"Средняя потеря на обучении: {train_loss:.4f}")

    val_loss, val_cer, val_wer, predictions, ground_truths = validate(
        model, val_loader, criterion, mapper, device
    )
    history["val_loss"].append(val_loss)
    history["val_cer"].append(val_cer)
    history["val_wer"].append(val_wer)

    print(f"Средняя потеря на валидации: {val_loss:.4f}")
    print(f"Средняя CER на валидации: {val_cer:.4f}")
    print(f"Средняя WER на валидации: {val_wer:.4f}")

    print("\nПримеры предсказаний (текущий вал):")
    shown = min(5, len(ground_truths), len(predictions))
    for i in range(shown):
        print(f"GT: '{ground_truths[i]}' -> Pred: '{predictions[i]}'")

    with torch.no_grad():
        fixed_outputs = model(fixed_images)
        fixed_preds = decode_predictions(fixed_outputs, mapper)

    print("\nФиксированные примеры для сравнения между эпохами:")
    fixed_shown = min(5, len(fixed_texts), len(fixed_preds))
    for i in range(fixed_shown):
        print(f"GT: '{fixed_texts[i]}' -> Pred: '{fixed_preds[i]}'")

    if val_cer < history["best_val_cer"]:
        history["best_val_cer"] = val_cer
        history["best_epoch"] = epoch
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "char_mapper": mapper,
                "history": history,
                "val_cer": val_cer,
                "val_wer": val_wer
            },
            best_checkpoint_path
        )
        print(f"Новая лучшая модель сохранена: {best_checkpoint_path} (CER: {val_cer:.4f})")
    else:
        epochs_no_improve += 1
        print(f"Нет улучшения CER. Эпох без улучшения: {epochs_no_improve}/{PATIENCE}")

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "char_mapper": mapper,
            "history": history,
        },
        last_checkpoint_path,
    )

    scheduler.step(val_loss)
    print(f"Текущий learning rate: {optimizer.param_groups[0]['lr']:.7f}")

    if epochs_no_improve >= PATIENCE:
        print("Раннее прекращение обучения из-за отсутствия улучшения CER.")
        break

print("\nОбучение завершено.")
print(f"Итоговый loss на обучении: {history['train_loss'][-1]:.4f}")
print(f"Итоговый loss на валидации: {history['val_loss'][-1]:.4f}")
print(f"Лучший CER на валидации: {history['best_val_cer']:.4f} (эпоха {history['best_epoch']})")
print(f"Итоговый WER на валидации: {history['val_wer'][-1]:.4f}")
print("="*30)

final_checkpoint_path = Path("crnn_checkpoint_last.pth")

torch.save(
    {
        "epoch": len(history["train_loss"]),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "char_mapper": mapper,
        "history": history,
    },
    final_checkpoint_path,
)
print(f"Последнее состояние модели сохранено в '{final_checkpoint_path}'")

###  Inference

In [ ]:
checkpoint_path = "crnn_ru_checkpoint_last.pth"

try:
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(checkpoint_path, map_location=device)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    mapper_for_val_infer = ckpt.get("char_mapper", mapper)
else:
    state_dict = ckpt
    mapper_for_val_infer = mapper

infer_model_for_val = CRNN(
    img_height=64,
    num_chars=mapper_for_val_infer.num_classes,
    hidden_size=256,
    num_layers=2,
).to(device)

infer_model_for_val.load_state_dict(state_dict)
infer_model_for_val.eval()

test_batch = next(iter(val_loader))

with torch.no_grad():
    images = test_batch['images'].to(device)
    outputs = infer_model_for_val(images)
    predictions = decode_predictions(outputs, mapper_for_val_infer)

fig, axes = plt.subplots(min(4, len(test_batch['texts'])), 1, 
                          figsize=(14, 3 * min(4, len(test_batch['texts']))))

if min(4, len(test_batch['texts'])) == 1:
    axes = [axes]

for i in range(min(4, len(test_batch['texts']))):
    ax = axes[i]
    
    img = test_batch['images'][i, 0].cpu().numpy()
    img = (img * 0.5) + 0.5
    img = np.clip(img, 0, 1)
    
    ax.imshow(img, cmap='gray')
    
    gt = test_batch['texts'][i]
    pred = predictions[i]
    
    title = f"GT:   {gt}\nPred: {pred}"
    color = 'green' if gt == pred else 'red'
    ax.set_title(title, fontsize=10, loc='left', color=color, weight='bold')
    ax.axis('off')

plt.suptitle('Примеры предсказаний на валидационном наборе', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

print(f"\n✓ Инференс завершён")

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageEnhance
import numpy as np
import torch
import matplotlib.pyplot as plt

checkpoint_path = "crnn_ru_checkpoint_last.pth"

try:
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(checkpoint_path, map_location=device)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    mapper_for_infer = ckpt.get("char_mapper", mapper)
else:
    state_dict = ckpt
    mapper_for_infer = mapper

infer_model = CRNN(
    img_height=64,
    num_chars=mapper_for_infer.num_classes,
    hidden_size=256,
    num_layers=2,
).to(device)

infer_model.load_state_dict(state_dict)
infer_model.eval()

CONTENT_CROP_THRESHOLD = 245
CONTENT_CROP_PADDING = 6

def enhance_handwriting_image(img):
    img = ImageOps.autocontrast(img, cutoff=1)
    img = ImageEnhance.Contrast(img).enhance(1.8)
    img = ImageEnhance.Brightness(img).enhance(1.12)
    return img

def build_binary_mask(img, threshold=CONTENT_CROP_THRESHOLD):
    img_arr = np.array(img, dtype=np.uint8)
    return np.where(img_arr > threshold, 255, 0).astype(np.uint8)

def compute_content_bbox_from_binary(binary_arr, padding=CONTENT_CROP_PADDING, background=255):
    foreground_coords = np.argwhere(binary_arr != background)
    if foreground_coords.size == 0:
        height, width = binary_arr.shape
        return (0, 0, width, height)

    y_min, x_min = foreground_coords.min(axis=0)
    y_max, x_max = foreground_coords.max(axis=0)

    x_min = max(0, int(x_min) - padding)
    y_min = max(0, int(y_min) - padding)
    x_max = min(binary_arr.shape[1], int(x_max) + padding + 1)
    y_max = min(binary_arr.shape[0], int(y_max) + padding + 1)
    return (x_min, y_min, x_max, y_max)

def crop_to_content(img, threshold=CONTENT_CROP_THRESHOLD, padding=CONTENT_CROP_PADDING):
    binary_arr = build_binary_mask(img, threshold=threshold)
    bbox = compute_content_bbox_from_binary(binary_arr, padding=padding)
    cropped_img = img.crop(bbox)
    cropped_binary = binary_arr[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    return cropped_img, cropped_binary, bbox

def build_tensor_from_pil(img, img_height=64, resample=Image.Resampling.LANCZOS):
    w, h = img.size
    new_w = max(4, int(round(img_height * w / max(h, 1))))
    img_resized = img.resize((new_w, img_height), resample)

    arr = np.array(img_resized, dtype=np.float32) / 255.0
    arr_norm = (arr - 0.5) / 0.5
    tensor = torch.from_numpy(arr_norm).unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    return tensor, img_resized, arr

def preprocess_one_image(
    image_path,
    img_height=64,
    crop_threshold=CONTENT_CROP_THRESHOLD,
    crop_padding=CONTENT_CROP_PADDING,
):
    img_original = Image.open(image_path).convert("L")
    img_enhanced = enhance_handwriting_image(img_original)
    img_cropped, cropped_binary, crop_bbox = crop_to_content(
        img_enhanced,
        threshold=crop_threshold,
        padding=crop_padding,
    )

    tensor, img_resized, arr = build_tensor_from_pil(img_cropped, img_height=img_height)
    return tensor, img_original, img_enhanced, img_cropped, img_resized, arr, cropped_binary, crop_bbox

def predict_from_pil(img, img_height=64, resample=Image.Resampling.LANCZOS):
    x, _, _ = build_tensor_from_pil(img, img_height=img_height, resample=resample)
    x = x.to(device)
    with torch.no_grad():
        outputs = infer_model(x)
        pred = decode_predictions(outputs, mapper_for_infer)[0]
    return pred

def show_preprocessing_stages(img_original, img_enhanced, img_cropped, img_resized):
    pred_original = predict_from_pil(img_original)
    pred_enhanced = predict_from_pil(img_enhanced)
    pred_cropped = predict_from_pil(img_cropped)
    pred_resized = predict_from_pil(img_resized)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    axes[0].imshow(img_original, cmap="gray")
    axes[0].set_title(f"Original\nPred: {pred_original}")
    axes[0].axis("off")

    axes[1].imshow(img_enhanced, cmap="gray")
    axes[1].set_title(f"Enhanced\nPred: {pred_enhanced}")
    axes[1].axis("off")

    axes[2].imshow(img_cropped, cmap="gray")
    axes[2].set_title(f"Cropped to content\nPred: {pred_cropped}")
    axes[2].axis("off")

    axes[3].imshow(img_resized, cmap="gray")
    axes[3].set_title(f"Final resize\nPred: {pred_resized}")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

def predict_text(image_path, show_preprocessing=False):
    x, img_original, img_enhanced, img_cropped, img_resized, _, _, _ = preprocess_one_image(image_path)

    if show_preprocessing:
        show_preprocessing_stages(img_original, img_enhanced, img_cropped, img_resized)

    x = x.to(device)
    with torch.no_grad():
        outputs = infer_model(x)  # [1, T, C]
        pred = decode_predictions(outputs, mapper_for_infer)[0]
    return pred

my_image = Path("test_pictures/will_be_good.jpg")
if not my_image.exists() and "data_samples" in globals() and len(data_samples) > 0:
    my_image = Path(data_samples[0]["image_path"])

if not my_image.exists():
    raise FileNotFoundError("Test image was not found. Set an existing path in my_image.")

pred_text = predict_text(my_image, show_preprocessing=True)

print("Image:", my_image)
print("Prediction:", pred_text)


In [ ]:
_, img_original, img_enhanced, img_cropped, img_resized, _, _, _ = preprocess_one_image(my_image)
show_preprocessing_stages(img_original, img_enhanced, img_cropped, img_resized)


In [ ]:
from PIL import ImageFilter

MASK_THRESHOLD = 185
MIN_COMPONENT_AREA = 10  # Remove tiny black blobs after thresholding.
THRESHOLD_UPSCALE_FACTOR = 2
THRESHOLD_MEDIAN_SIZE = 3
WORD_GAP_MIN_WIDTH = 6
MAX_WORDS_PER_CHUNK = 2
WORD_CHUNK_PADDING = 4

examples_checkpoint_path = Path("crnn_ru_checkpoint_last.pth")
examples_dir = Path("data/inference_examples")

if not examples_checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {examples_checkpoint_path}")

if not examples_dir.exists():
    raise FileNotFoundError(f"Examples directory not found: {examples_dir}")

example_paths = sorted(
    path for path in examples_dir.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

if not example_paths:
    raise FileNotFoundError(f"No supported images found in {examples_dir}")

try:
    examples_ckpt = torch.load(examples_checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    examples_ckpt = torch.load(examples_checkpoint_path, map_location=device)

if isinstance(examples_ckpt, dict) and "model_state_dict" in examples_ckpt:
    examples_state_dict = examples_ckpt["model_state_dict"]
    examples_mapper = examples_ckpt.get("char_mapper", mapper)
else:
    examples_state_dict = examples_ckpt
    examples_mapper = mapper

examples_model = CRNN(
    img_height=64,
    num_chars=examples_mapper.num_classes,
    hidden_size=256,
    num_layers=2,
).to(device)

examples_model.load_state_dict(examples_state_dict)
examples_model.eval()

def tensor_from_resized_pil(img):
    arr = np.array(img, dtype=np.float32) / 255.0
    arr_norm = (arr - 0.5) / 0.5
    return torch.from_numpy(arr_norm).unsqueeze(0).unsqueeze(0)

def predict_from_resized_pil(img):
    x = tensor_from_resized_pil(img).to(device)
    with torch.no_grad():
        outputs = examples_model(x)
        pred = decode_predictions(outputs, examples_mapper)[0]
    return pred

def resize_for_examples_from_pil(img, img_height=64, resample=Image.Resampling.NEAREST):
    _, resized_img, _ = build_tensor_from_pil(img, img_height=img_height, resample=resample)
    return resized_img

def predict_with_examples_from_pil(img, img_height=64, resample=Image.Resampling.NEAREST):
    resized_img = resize_for_examples_from_pil(img, img_height=img_height, resample=resample)
    return resized_img, predict_from_resized_pil(resized_img)

def remove_small_black_components(binary_arr, min_area=0):
    if min_area <= 0:
        return binary_arr

    foreground = binary_arr == 0
    visited = np.zeros_like(foreground, dtype=bool)
    height, width = foreground.shape
    cleaned = binary_arr.copy()

    for y in range(height):
        for x in range(width):
            if visited[y, x] or not foreground[y, x]:
                continue

            stack = [(y, x)]
            visited[y, x] = True
            component = []

            while stack:
                cy, cx = stack.pop()
                component.append((cy, cx))

                for ny in range(max(0, cy - 1), min(height, cy + 2)):
                    for nx in range(max(0, cx - 1), min(width, cx + 2)):
                        if visited[ny, nx] or not foreground[ny, nx]:
                            continue
                        visited[ny, nx] = True
                        stack.append((ny, nx))

            if len(component) < min_area:
                for cy, cx in component:
                    cleaned[cy, cx] = 255

    return cleaned

def crop_binary_pil_to_content(img, padding=0):
    binary_arr = np.array(img, dtype=np.uint8)
    bbox = compute_content_bbox_from_binary(binary_arr, padding=padding)
    return img.crop(bbox), bbox

def find_word_bounds(binary_arr, min_gap_width=WORD_GAP_MIN_WIDTH):
    has_foreground = np.any(binary_arr == 0, axis=0)
    content_cols = np.flatnonzero(has_foreground)
    if content_cols.size == 0:
        return [(0, binary_arr.shape[1])]

    left = int(content_cols[0])
    right = int(content_cols[-1])
    effective_min_gap = max(min_gap_width, int(round(binary_arr.shape[0] * 0.01)))
    min_word_width = max(4, int(round(binary_arr.shape[0] * 0.10)))

    bounds = []
    segment_start = left
    x = left

    while x <= right:
        if has_foreground[x]:
            x += 1
            continue

        gap_start = x
        while x <= right and not has_foreground[x]:
            x += 1
        gap_end = x - 1

        if gap_end - gap_start + 1 >= effective_min_gap:
            split_x = (gap_start + gap_end) // 2
            if split_x - segment_start >= min_word_width:
                bounds.append((segment_start, split_x))
            segment_start = split_x + 1

    if right + 1 - segment_start >= min_word_width:
        bounds.append((segment_start, right + 1))

    if not bounds:
        return [(left, right + 1)]
    return bounds

def group_word_bounds(word_bounds, max_words_per_chunk=MAX_WORDS_PER_CHUNK, padding=WORD_CHUNK_PADDING, max_width=None):
    if max_width is None:
        max_width = word_bounds[-1][1]

    chunk_bounds = []
    for idx in range(0, len(word_bounds), max_words_per_chunk):
        chunk_words = word_bounds[idx:idx + max_words_per_chunk]
        start = max(0, chunk_words[0][0] - padding)
        end = min(max_width, chunk_words[-1][1] + padding)
        chunk_bounds.append((start, end))
    return chunk_bounds

def stitch_chunk_previews(chunk_images, gap=12, background=255):
    if not chunk_images:
        return Image.new("L", (32, 32), color=background)

    max_height = max(img.height for img in chunk_images)
    total_width = sum(img.width for img in chunk_images) + gap * max(0, len(chunk_images) - 1)
    canvas = Image.new("L", (total_width, max_height), color=background)

    x_offset = 0
    for chunk_img in chunk_images:
        y_offset = (max_height - chunk_img.height) // 2
        canvas.paste(chunk_img, (x_offset, y_offset))
        x_offset += chunk_img.width + gap

    return canvas

def segment_thresholded_words(
    thresholded_img,
    min_gap_width=WORD_GAP_MIN_WIDTH,
    max_words_per_chunk=MAX_WORDS_PER_CHUNK,
    chunk_padding=WORD_CHUNK_PADDING,
):
    binary_arr = np.array(thresholded_img, dtype=np.uint8)
    word_bounds = find_word_bounds(binary_arr, min_gap_width=min_gap_width)
    chunk_bounds = group_word_bounds(
        word_bounds,
        max_words_per_chunk=max_words_per_chunk,
        padding=chunk_padding,
        max_width=thresholded_img.width,
    )
    chunk_images = []
    for start, end in chunk_bounds:
        raw_chunk = thresholded_img.crop((start, 0, end, thresholded_img.height))
        cropped_chunk, _ = crop_binary_pil_to_content(raw_chunk, padding=max(1, chunk_padding // 2))
        chunk_images.append(cropped_chunk)

    return {
        "binary_arr": binary_arr,
        "word_bounds": word_bounds,
        "chunk_bounds": chunk_bounds,
        "chunk_images": chunk_images,
    }

def make_thresholded_preprocessing(
    image_path,
    threshold=MASK_THRESHOLD,
    min_component_area=MIN_COMPONENT_AREA,
    upscale_factor=THRESHOLD_UPSCALE_FACTOR,
    median_size=THRESHOLD_MEDIAN_SIZE,
    content_padding=CONTENT_CROP_PADDING,
    return_debug=False,
):
    _, _, img_enhanced, _, baseline_img, _, _, _ = preprocess_one_image(image_path)
    working_img = img_enhanced

    if upscale_factor > 1:
        working_img = img_enhanced.resize(
            (img_enhanced.width * upscale_factor, img_enhanced.height * upscale_factor),
            Image.Resampling.LANCZOS,
        )

    if median_size and median_size > 1:
        if median_size % 2 == 0:
            median_size += 1
        working_img = working_img.filter(ImageFilter.MedianFilter(size=median_size))

    working_arr = np.array(working_img, dtype=np.uint8)
    binary_arr = np.where(working_arr > threshold, 255, 0).astype(np.uint8)
    scaled_min_area = int(min_component_area * max(1, upscale_factor) * max(1, upscale_factor))
    cleaned_arr = remove_small_black_components(binary_arr, min_area=scaled_min_area)
    thresholded_upscaled_img = Image.fromarray(cleaned_arr, mode="L")

    content_bbox = compute_content_bbox_from_binary(cleaned_arr, padding=content_padding)
    thresholded_cropped_img = thresholded_upscaled_img.crop(content_bbox)
    thresholded_cropped_img, tight_bbox = crop_binary_pil_to_content(
        thresholded_cropped_img,
        padding=max(1, content_padding // 2),
    )
    _, thresholded_model_img, _ = build_tensor_from_pil(
        thresholded_cropped_img,
        img_height=baseline_img.height,
        resample=Image.Resampling.NEAREST,
    )

    result = {
        "baseline_img": baseline_img,
        "working_img": working_img,
        "thresholded_upscaled_img": thresholded_upscaled_img,
        "thresholded_cropped_img": thresholded_cropped_img,
        "thresholded_model_img": thresholded_model_img,
        "content_bbox": content_bbox,
        "tight_bbox": tight_bbox,
    }
    if return_debug:
        return result
    return baseline_img, thresholded_model_img

def predict_segmented_threshold_text(
    image_path,
    threshold=MASK_THRESHOLD,
    min_component_area=MIN_COMPONENT_AREA,
    min_gap_width=WORD_GAP_MIN_WIDTH,
    max_words_per_chunk=MAX_WORDS_PER_CHUNK,
    chunk_padding=WORD_CHUNK_PADDING,
    return_debug=False,
):
    result = make_thresholded_preprocessing(
        image_path,
        threshold=threshold,
        min_component_area=min_component_area,
        return_debug=True,
    )
    segment_info = segment_thresholded_words(
        result["thresholded_cropped_img"],
        min_gap_width=min_gap_width,
        max_words_per_chunk=max_words_per_chunk,
        chunk_padding=chunk_padding,
    )

    chunk_model_imgs = []
    chunk_predictions = []
    for chunk_img in segment_info["chunk_images"]:
        chunk_model_img, chunk_prediction = predict_with_examples_from_pil(
            chunk_img,
            img_height=result["baseline_img"].height,
            resample=Image.Resampling.NEAREST,
        )
        chunk_model_imgs.append(chunk_model_img)
        chunk_predictions.append(chunk_prediction.strip())

    merged_prediction = " ".join(pred for pred in chunk_predictions if pred)
    if not merged_prediction:
        fallback_prediction = predict_from_resized_pil(result["thresholded_model_img"])
        chunk_predictions = [fallback_prediction]
        chunk_model_imgs = [result["thresholded_model_img"]]
        segment_info["word_bounds"] = [(0, result["thresholded_cropped_img"].width)]
        segment_info["chunk_bounds"] = [(0, result["thresholded_cropped_img"].width)]
        segment_info["chunk_images"] = [result["thresholded_cropped_img"]]
        merged_prediction = fallback_prediction

    result.update(segment_info)
    result["chunk_model_imgs"] = chunk_model_imgs
    result["chunk_predictions"] = chunk_predictions
    result["merged_prediction"] = merged_prediction
    result["chunk_preview_strip"] = stitch_chunk_previews(result["chunk_images"])

    if return_debug:
        return result
    return result["baseline_img"], result["thresholded_model_img"], merged_prediction

fig, axes = plt.subplots(len(example_paths), 4, figsize=(28, 4 * len(example_paths)))
axes = np.atleast_2d(axes)

for row_idx, image_path in enumerate(example_paths):
    segmented_result = predict_segmented_threshold_text(image_path, return_debug=True)
    baseline_pred = predict_from_resized_pil(segmented_result["baseline_img"])
    chunk_summary = " | ".join(segmented_result["chunk_predictions"])

    axes[row_idx, 0].imshow(segmented_result["baseline_img"], cmap="gray")
    axes[row_idx, 0].set_title(
        f"{image_path.name}\nBaseline: {baseline_pred}",
        loc="left",
        fontsize=10,
        weight="bold",
    )
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(segmented_result["working_img"], cmap="gray")
    axes[row_idx, 1].set_title(
        "Full enhanced frame -> upscale -> smooth",
        loc="left",
        fontsize=10,
        weight="bold",
    )
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(segmented_result["thresholded_cropped_img"], cmap="gray")
    for start, end in segmented_result["word_bounds"]:
        axes[row_idx, 2].axvline(start, color="deepskyblue", linestyle="--", linewidth=1)
        axes[row_idx, 2].axvline(end, color="deepskyblue", linestyle="--", linewidth=1)
    axes[row_idx, 2].set_title(
        "Thresholded + cropped\nBlue lines = exact empty-column word bounds",
        loc="left",
        fontsize=10,
        weight="bold",
    )
    axes[row_idx, 2].axis("off")

    axes[row_idx, 3].imshow(segmented_result["chunk_preview_strip"], cmap="gray")
    axes[row_idx, 3].set_title(
        f"Raw chunk previews\nMerged OCR: {segmented_result['merged_prediction']}\nChunks: {chunk_summary}",
        loc="left",
        fontsize=10,
        weight="bold",
    )
    axes[row_idx, 3].axis("off")

plt.tight_layout()
plt.show()

print(f"Checkpoint: {examples_checkpoint_path}")
print(f"Examples shown: {len(example_paths)}")
print(
    f"MASK_THRESHOLD={MASK_THRESHOLD}, MIN_COMPONENT_AREA={MIN_COMPONENT_AREA}, "
    f"THRESHOLD_UPSCALE_FACTOR={THRESHOLD_UPSCALE_FACTOR}, THRESHOLD_MEDIAN_SIZE={THRESHOLD_MEDIAN_SIZE}, "
    f"WORD_GAP_MIN_WIDTH={WORD_GAP_MIN_WIDTH}, MAX_WORDS_PER_CHUNK={MAX_WORDS_PER_CHUNK}"
)


In [ ]:
selected_image_for_emotion = Path("data/inference_examples/happy.jpg")
emotion_notebook_path = Path("emotion_analysis.ipynb")
emotion_checkpoint_path = Path("crnn_ru_checkpoint_last.pth")

if not selected_image_for_emotion.exists():
    if "my_image" in globals() and Path(my_image).exists():
        selected_image_for_emotion = Path(my_image)
    else:
        raise FileNotFoundError(f"Selected image not found: {selected_image_for_emotion}")

if not emotion_notebook_path.exists():
    raise FileNotFoundError(f"Emotion notebook not found: {emotion_notebook_path}")

if not emotion_checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {emotion_checkpoint_path}")

import json

emotion_nb = json.loads(emotion_notebook_path.read_text(encoding="utf-8"))
emotion_ns = {"__name__": "__emotion_analysis_import__"}

for cell in emotion_nb["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if source.lstrip().startswith("examples = ["):
        break

    exec(source, emotion_ns)

emotion_analyzer = emotion_ns["analyzer"]
build_emotion_distribution_fn = emotion_ns["build_emotion_distribution"]
plot_emotion_trace_fn = emotion_ns["plot_emotion_trace"]
emotion_mask_threshold = 150
emotion_min_component_area = 10
emotion_word_gap_min_width = 1
emotion_max_words_per_chunk = 2

if "examples_model" not in globals() or "examples_mapper" not in globals():
    try:
        emotion_ckpt = torch.load(emotion_checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        emotion_ckpt = torch.load(emotion_checkpoint_path, map_location=device)

    if isinstance(emotion_ckpt, dict) and "model_state_dict" in emotion_ckpt:
        emotion_state_dict = emotion_ckpt["model_state_dict"]
        examples_mapper = emotion_ckpt.get("char_mapper", mapper)
    else:
        emotion_state_dict = emotion_ckpt
        examples_mapper = mapper

    examples_model = CRNN(
        img_height=64,
        num_chars=examples_mapper.num_classes,
        hidden_size=256,
        num_layers=2,
    ).to(device)
    examples_model.load_state_dict(emotion_state_dict)
    examples_model.eval()

def tensor_from_pil_for_emotion(img):
    arr = np.array(img, dtype=np.float32) / 255.0
    arr_norm = (arr - 0.5) / 0.5
    return torch.from_numpy(arr_norm).unsqueeze(0).unsqueeze(0)

def predict_resized_image_for_emotion(img):
    if "predict_from_resized_pil" in globals():
        return predict_from_resized_pil(img)

    x = tensor_from_pil_for_emotion(img).to(device)
    with torch.no_grad():
        outputs = examples_model(x)
        raw_prediction = decode_predictions(outputs, examples_mapper)[0]
    return raw_prediction

def predict_for_emotion(
    image_path,
    threshold=emotion_mask_threshold,
    min_component_area=emotion_min_component_area,
    min_gap_width=emotion_word_gap_min_width,
    max_words_per_chunk=emotion_max_words_per_chunk,
):
    _, _, _, _, baseline_img, _, _, _ = preprocess_one_image(image_path)
    baseline_prediction = predict_resized_image_for_emotion(baseline_img)

    if "predict_segmented_threshold_text" in globals():
        threshold_result = predict_segmented_threshold_text(
            image_path,
            threshold=threshold,
            min_component_area=min_component_area,
            min_gap_width=min_gap_width,
            max_words_per_chunk=max_words_per_chunk,
            return_debug=True,
        )
    else:
        _, thresholded_img = make_thresholded_preprocessing(
            image_path,
            threshold=threshold,
            min_component_area=min_component_area,
        )
        threshold_prediction = predict_resized_image_for_emotion(thresholded_img)
        threshold_result = {
            "thresholded_cropped_img": thresholded_img,
            "thresholded_model_img": thresholded_img,
            "word_bounds": [(0, thresholded_img.width)],
            "chunk_bounds": [(0, thresholded_img.width)],
            "chunk_predictions": [threshold_prediction],
            "chunk_model_imgs": [thresholded_img],
            "chunk_images": [thresholded_img],
            "merged_prediction": threshold_prediction,
        }

    return baseline_img, baseline_prediction, threshold_result

emotion_baseline_img, baseline_ocr_prediction, threshold_result = predict_for_emotion(
    selected_image_for_emotion,
)
emotion_thresholded_preview_img = threshold_result["thresholded_cropped_img"]
thresholded_ocr_prediction = threshold_result["merged_prediction"]

baseline_emotion_result = emotion_analyzer.analyze(baseline_ocr_prediction)
baseline_positive_scores, baseline_emotion_distribution, baseline_dominant_emotion = build_emotion_distribution_fn(baseline_emotion_result)

thresholded_emotion_result = emotion_analyzer.analyze(thresholded_ocr_prediction)
thresholded_positive_scores, thresholded_emotion_distribution, thresholded_dominant_emotion = build_emotion_distribution_fn(thresholded_emotion_result)

fig, axes = plt.subplots(1, 2, figsize=(18, 4))
axes[0].imshow(emotion_baseline_img, cmap="gray")
axes[0].set_title(
    f"Baseline preprocessing\nOCR: {baseline_ocr_prediction}",
    loc="left",
    fontsize=11,
    weight="bold",
)
axes[0].axis("off")

axes[1].imshow(emotion_thresholded_preview_img, cmap="gray")
for start, end in threshold_result["chunk_bounds"]:
    axes[1].axvline(start, color="deepskyblue", linestyle="--", linewidth=1)
    axes[1].axvline(end, color="deepskyblue", linestyle="--", linewidth=1)
axes[1].set_title(
    f"Thresholded + segmented\nOCR: {thresholded_ocr_prediction}",
    loc="left",
    fontsize=11,
    weight="bold",
)
axes[1].axis("off")
plt.tight_layout()
plt.show()

if threshold_result["chunk_images"]:
    fig, chunk_axes = plt.subplots(1, len(threshold_result["chunk_images"]), figsize=(4 * len(threshold_result["chunk_images"]), 4))
    chunk_axes = np.atleast_1d(chunk_axes)
    for idx, (ax, chunk_img, chunk_pred) in enumerate(zip(chunk_axes, threshold_result["chunk_images"], threshold_result["chunk_predictions"]), start=1):
        ax.imshow(chunk_img, cmap="gray")
        ax.set_title(f"Chunk {idx}\n{chunk_pred}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Image: {selected_image_for_emotion}")
print("=" * 80)
print("BASELINE")
print(f"OCR prediction: {baseline_ocr_prediction}")
print(f"Normalized text: {baseline_emotion_result.normalized_text}")
print(f"Corrected text: {baseline_emotion_result.corrected_text}")
print(f"Token count: {baseline_emotion_result.content_token_count}")
print(f"Enough text: {baseline_emotion_result.enough_text}")
print(f"Dominant emotion: {baseline_dominant_emotion}")
print(f"Distribution: {baseline_emotion_distribution}")
print(f"Positive scores: {baseline_positive_scores}")
print()
print("THRESHOLDED + SEGMENTED")
print(f"Word bounds: {threshold_result['word_bounds']}")
print(f"Chunk bounds: {threshold_result['chunk_bounds']}")
print(f"Chunk predictions: {threshold_result['chunk_predictions']}")
print(f"OCR prediction: {thresholded_ocr_prediction}")
print(f"Normalized text: {thresholded_emotion_result.normalized_text}")
print(f"Corrected text: {thresholded_emotion_result.corrected_text}")
print(f"Token count: {thresholded_emotion_result.content_token_count}")
print(f"Enough text: {thresholded_emotion_result.enough_text}")
print(f"Dominant emotion: {thresholded_dominant_emotion}")
print(f"Distribution: {thresholded_emotion_distribution}")
print(f"Positive scores: {thresholded_positive_scores}")

plot_emotion_trace_fn(
    baseline_emotion_result,
    title=f"Baseline emotion trace for {selected_image_for_emotion.name}",
)

plot_emotion_trace_fn(
    thresholded_emotion_result,
    title=f"Thresholded emotion trace for {selected_image_for_emotion.name}",
)
